In [1]:
! pip -q install timm
! git clone https://github.com/mv-lab/swin2sr.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 12.8 MB/s eta 0:00:00
Cloning into 'swin2sr'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 180 (delta 18), reused 16 (delta 11), pack-reused 148
Receiving objects: 100% (180/180), 20.54 MiB | 41.82 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

os.chdir("./swin2sr")

In [4]:
def load_img (filename, debug=False, norm=True, resize=None):
    img = cv2.imread(filename)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if norm:
        img = img / 255.
        img = img.astype(np.float32)
    if debug:
        print (img.shape, img.dtype, img.min(), img.max())

    if resize:
        img = cv2.resize(img, (resize[0], resize[1]))

    return img

def plot_all (images, axis='off', figsize=(16, 8)):

    fig = plt.figure(figsize=figsize, dpi=80)
    nplots = len(images)
    for i in range(nplots):
        plt.subplot(1,nplots,i+1)
        plt.axis(axis)
        plt.imshow(images[i])
    plt.show()

In [5]:
import os
from PIL import Image

# Function to split the image into patches and save them in a folder
def split_image_and_save(image_path, patch_size, output_folder):
    image = Image.open(image_path).convert('L')  # Convert to grayscale
    width, height = image.size
    patches = []

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for i in range(0, height, patch_size):
        for j in range(0, width, patch_size):
            box = (j, i, j + patch_size, i + patch_size)
            patch = image.crop(box)
            patch_filename = f'{output_folder}/patch_{i}_{j}.png'
            patch.save(patch_filename)
            patches.append(patch_filename)

    return patches, image.size



In [6]:
def stitch_images(patches_folder, original_size, patch_size, output_image_path):
    width, height = original_size
    super_res_image = Image.new('L', (width * 4, height * 4))  # 4x super-resolution

    for i in range(0, height, patch_size):
        for j in range(0, width, patch_size):
            patch_filename = f'{patches_folder}/patch_{i}_{j}_Swin2SR.png'
            patch = Image.open(patch_filename)
            super_res_image.paste(patch, (j * 4, i * 4))

    super_res_image.save(output_image_path,'PNG')

In [7]:
# Define the folder containing images
image_folder = '/content/drive/MyDrive/coal-mineral-dataset/images'  # Change this to your folder path

# Get the list of all images in the folder
image_files = [f for f in os.listdir(image_folder) if f.endswith('.bmp')]

# Change directory to testsets
os.chdir("./testsets")

# Clean and create real-inputs directory from scratch
!rm -r real-inputs
!mkdir real-inputs

# Change back to the parent directory
os.chdir("..")


In [ ]:
# Loop through each image in the folder
for image_file in image_files:
    image_path = os.path.join(image_folder, image_file)

    output_folder = '/content/swin2sr/testsets/real-inputs'
    split_image_and_save(image_path, 200, output_folder)

    # Clean and create the inputs/ directory from scratch
    !rm -r inputs
    !mkdir inputs

    # Put some images into inputs/
    !cp testsets/real-inputs/* inputs/

    # Check the images in input/
    !ls inputs

    # Run the super-resolution script
    !python main_test_swin2sr.py --task compressed_sr --scale 4 --training_patch_size 48 --model_path model_zoo/swin2sr/Swin2SR_CompressedSR_X4_48.pth --folder_lq ./inputs/ --save_img_only

    patch_folder = '/content/swin2sr/results/swin2sr_compressed_sr_x4'
    image = Image.open(image_path).convert('L')  # Convert to grayscale
    width, height = image.size
    original_size = [width, height]
    patch_size = 200
    scale_factor = 4

    # Save the final stitched image with the same name as the original image
    output_image_path = os.path.join('/content/swin2sr/results', image_file)
    stitch_images(patch_folder, original_size, 200, output_image_path)

rm: cannot remove 'inputs': No such file or directory
patch_0_0.png	     patch_1200_0.png	  patch_200_0.png     patch_600_0.png
patch_0_1000.png     patch_1200_1000.png  patch_200_1000.png  patch_600_1000.png
patch_0_1200.png     patch_1200_1200.png  patch_200_1200.png  patch_600_1200.png
patch_0_1400.png     patch_1200_1400.png  patch_200_1400.png  patch_600_1400.png
patch_0_1600.png     patch_1200_1600.png  patch_200_1600.png  patch_600_1600.png
patch_0_1800.png     patch_1200_1800.png  patch_200_1800.png  patch_600_1800.png
patch_0_2000.png     patch_1200_2000.png  patch_200_2000.png  patch_600_2000.png
patch_0_200.png      patch_1200_200.png   patch_200_200.png   patch_600_200.png
patch_0_400.png      patch_1200_400.png   patch_200_400.png   patch_600_400.png
patch_0_600.png      patch_1200_600.png   patch_200_600.png   patch_600_600.png
patch_0_800.png      patch_1200_800.png   patch_200_800.png   patch_600_800.png
patch_1000_0.png     patch_1400_0.png	  patch_400_0.png     patch_